# Pedestrian Evacuation Simulation, Version 2

**Factorial computational study with two exit-flow models (research prototype)**
Author: Christian Somtoo Obiechina

**Status:** Synthetic simulation output. Neither model is calibrated or validated against observed evacuation data. Results describe these models under their stated assumptions.

## What changed from version 1

1. **Two exit-flow models.** The version 1 model had no physical limit on how fast people could leave through the exit, which the verification checks below show it exceeded in 99.8% of runs. The new `capacity` model serves the exit queue at an assumed flow rate, so the width effect has a physical basis. The `unconstrained` model is kept for comparison.
2. **More replicates.** 20 seeded replicates per condition, up from 10.
3. **Verification checks.** Reproducibility, boundary invariant, count and horizon checks, and a flow-limit check for each model.
4. **Sensitivity analysis.** The width effect is re-estimated under alternative repulsion and separation-radius settings, so conclusions do not rest on one arbitrary constant.
5. **Effect sizes with bootstrap confidence intervals**, reported alongside the p-values, and Holm correction across all six paired tests.

## Important cautions

- The flow rate `EXIT_FLOW_J` is an **assumption** that must be checked against a cited source before publication.
- Replicates vary only in agent placement and individual speed, so simulated variation is small. Confidence intervals and p-values describe this model's variation, not real-world uncertainty.
- The capacity model satisfies its flow limit **by construction**. The verification check confirms the implementation, not the realism of the assumption.


## 1. Setup, design and parameters

In [1]:
import json
import math
import platform
from collections import deque
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


In [2]:
# ----------------------------------------------------------------------------
# 1. Experimental design and parameters
# ----------------------------------------------------------------------------
CROWD_SIZES = [40, 80]            # agents
EXIT_WIDTHS = [0.8, 1.2, 1.6]     # metres
DESIRED_SPEEDS = [1.0, 1.3]       # m/s (mean of the agent distribution)
OBSTACLE = [False, True]          # obstacle in front of the exit
MODELS = ["unconstrained", "capacity"]
REPLICATES = 20                   # seeded replicates per condition
BASE_SEED = 20260101

ROOM = 10.0                       # square room, 10 m x 10 m
EXIT_X = 5.0                      # exit centred on the top wall (y = 10)
DT = 0.05                         # time step, seconds
T_MAX = 120.0                     # horizon, seconds

# Inter-agent interaction (baseline values, varied in the sensitivity analysis)
REPULSION = 1.5                   # m/s
SEP_RADIUS = 0.8                  # m
MAX_SPEED = 2.0                   # m/s speed cap

# Obstacle
OBS_CENTRE = (5.0, 8.2)
OBS_RADIUS = 0.5
OBS_INFLUENCE = 0.85

# Exit flow assumption for the capacity model (persons per metre of width per second).
# Literature values for bottleneck flow are often quoted around 1.0 to 1.5. This is a
# modelling assumption and must be checked against a cited source before publication.
EXIT_FLOW_J = 1.3

SENSITIVITY_VARIANTS = [
    # name, repulsion, separation radius
    ("baseline", 1.5, 0.8),
    ("repulsion_low", 0.75, 0.8),
    ("repulsion_high", 3.0, 0.8),
    ("radius_small", 1.5, 0.6),
    ("radius_large", 1.5, 1.0),
]


In [3]:
# ----------------------------------------------------------------------------
# 2. Agent model
# ----------------------------------------------------------------------------
def place_agents(n, rng, max_tries=20000):
    """Place n agents in the lower part of the room with a minimum separation of 0.4 m."""
    pts = []
    tries = 0
    while len(pts) < n:
        tries += 1
        if tries > max_tries:
            raise RuntimeError("Could not place agents; reduce crowd size.")
        cand = np.array([rng.uniform(0.5, ROOM - 0.5), rng.uniform(0.5, 5.5)])
        if all(np.hypot(*(cand - p)) >= 0.4 for p in pts):
            pts.append(cand)
    return np.array(pts)


def simulate(n, width, desired_speed, obstacle, seed, model="unconstrained",
             repulsion=REPULSION, sep_radius=SEP_RADIUS):
    """Run one evacuation. Returns per-agent evacuation times (NaN if not evacuated)."""
    if model not in MODELS:
        raise ValueError(f"Unknown model: {model}")

    rng = np.random.default_rng(seed)
    pos = place_agents(n, rng)
    v_des = np.clip(rng.normal(desired_speed, 0.1, n), 0.5, 1.8)
    active = np.ones(n, dtype=bool)     # not yet evacuated
    queued = np.zeros(n, dtype=bool)    # waiting at the exit (capacity model only)
    evac_time = np.full(n, np.nan)
    queue = deque()
    credit = 0.0                        # unused exit capacity, in persons
    half = width / 2.0
    t = 0.0

    while t < T_MAX and active.any():
        moving = np.where(active & ~queued)[0]
        arrivals = np.array([], dtype=int)

        if moving.size:
            p = pos[moving]
            gx = np.clip(p[:, 0], EXIT_X - half + 0.15, EXIT_X + half - 0.15)
            goal = np.column_stack([gx, np.full(moving.size, ROOM)])
            delta = goal - p
            dist = np.linalg.norm(delta, axis=1)
            unit = delta / np.maximum(dist, 1e-9)[:, None]
            v = v_des[moving][:, None] * unit

            if moving.size > 1:
                diff = p[:, None, :] - p[None, :, :]
                r = np.linalg.norm(diff, axis=2)
                np.fill_diagonal(r, np.inf)
                close = r < sep_radius
                if close.any():
                    mag = np.where(close, repulsion * (sep_radius - r) / sep_radius, 0.0)
                    safe_r = np.where(np.isfinite(r), r, 1.0)
                    v += (mag[:, :, None] * (diff / safe_r[:, :, None])).sum(axis=1)

            if obstacle:
                c = np.array(OBS_CENTRE)
                dc = p - c
                rc = np.linalg.norm(dc, axis=1)
                near = rc < OBS_RADIUS + OBS_INFLUENCE
                push = np.where(near, 2.0 * (OBS_RADIUS + OBS_INFLUENCE - rc) / OBS_INFLUENCE, 0.0)
                v += push[:, None] * dc / np.maximum(rc, 1e-9)[:, None]

            s = np.linalg.norm(v, axis=1)
            v *= np.minimum(1.0, MAX_SPEED / np.maximum(s, 1e-9))[:, None]

            new_p = p + v * DT
            new_p[:, 0] = np.clip(new_p[:, 0], 0.2, ROOM - 0.2)
            new_p[:, 1] = np.clip(new_p[:, 1], 0.2, ROOM)
            pos[moving] = new_p

            # Verification invariant: agents stay inside the room
            assert np.all((new_p >= 0) & (new_p <= ROOM)), "agent left the room"

            at_zone = (new_p[:, 1] >= ROOM - 1e-9) & (np.abs(new_p[:, 0] - EXIT_X) <= half)
            arrivals = moving[at_zone]

        t += DT

        if model == "unconstrained":
            evac_time[arrivals] = t
            active[arrivals] = False
        else:
            for a in arrivals:
                queue.append(a)
                queued[a] = True
            credit += EXIT_FLOW_J * width * DT
            if not queue:
                credit = min(credit, 1.0)   # unused capacity does not accumulate
            while queue and credit >= 1.0:
                a = queue.popleft()
                evac_time[a] = t
                active[a] = False
                queued[a] = False
                credit -= 1.0

    return evac_time


def summarise_run(evac_time, n):
    """Outcome measures for one run."""
    evacuated = np.isfinite(evac_time)
    n_evac = int(evacuated.sum())
    times = np.sort(evac_time[evacuated])
    target = math.ceil(0.9 * n)
    if n_evac >= target:
        t90, censored = float(times[target - 1]), False
    else:
        t90, censored = T_MAX, True          # right-censored at the horizon
    return {
        "t90_s": t90,
        "t90_censored": censored,
        "fraction_evacuated": n_evac / n,
        "mean_evacuation_time_s": float(times.mean()) if n_evac else np.nan,
        "throughput_per_s": n_evac / times[-1] if n_evac else 0.0,
        "n_evacuated": n_evac,
        "last_evacuation_s": float(times[-1]) if n_evac else np.nan,
    }


In [4]:
# ----------------------------------------------------------------------------
# 3. Experiment
# ----------------------------------------------------------------------------
def run_experiment():
    rows, agent_rows = [], []
    conditions = [(m, n, w, v, o) for m in MODELS for n in CROWD_SIZES
                  for w in EXIT_WIDTHS for v in DESIRED_SPEEDS for o in OBSTACLE]
    for cond_id, (m, n, w, v, o) in enumerate(conditions):
        for rep in range(REPLICATES):
            seed = BASE_SEED + rep          # same seed per replicate: paired comparisons
            evac_time = simulate(n, w, v, o, seed, model=m)
            rows.append({
                "condition_id": cond_id, "model": m, "n_agents": n,
                "exit_width_m": w, "desired_speed_mps": v, "obstacle": int(o),
                "replicate": rep, "seed": seed, **summarise_run(evac_time, n),
            })
    return pd.DataFrame(rows)


def run_sensitivity():
    """One-at-a-time sensitivity of the width effect to the interaction parameters."""
    rows = []
    for name, rep_force, radius in SENSITIVITY_VARIANTS:
        for model in MODELS:
            t_narrow, t_wide = [], []
            for rep in range(REPLICATES):
                seed = BASE_SEED + rep
                for w, bucket in [(0.8, t_narrow), (1.6, t_wide)]:
                    et = simulate(80, w, 1.3, False, seed, model=model,
                                  repulsion=rep_force, sep_radius=radius)
                    bucket.append(summarise_run(et, 80)["t90_s"])
            diff = np.array(t_wide) - np.array(t_narrow)   # 1.6 m minus 0.8 m
            lo, hi = bootstrap_median_ci(diff)
            rows.append({
                "variant": name, "repulsion_mps": rep_force, "sep_radius_m": radius,
                "model": model, "median_width_effect_s": float(np.median(diff)),
                "ci_low_s": lo, "ci_high_s": hi,
                "median_t90_narrow_s": float(np.median(t_narrow)),
                "median_t90_wide_s": float(np.median(t_wide)),
            })
    return pd.DataFrame(rows)


In [5]:
# ----------------------------------------------------------------------------
# 4. Verification checks
# ----------------------------------------------------------------------------
def run_verification(runs):
    checks = []

    def add(name, model, value, passed, note=""):
        checks.append({"check": name, "model": model, "value": value,
                       "passed": bool(passed), "note": note})

    # Reproducibility: identical seed gives identical output
    a = simulate(40, 1.2, 1.3, True, BASE_SEED, model="capacity")
    b = simulate(40, 1.2, 1.3, True, BASE_SEED, model="capacity")
    add("reproducibility (same seed, same output)", "capacity",
        bool(np.array_equal(a, b, equal_nan=True)),
        np.array_equal(a, b, equal_nan=True))

    # Edge case: a crowd of one agent evacuates
    single = simulate(1, 1.2, 1.3, False, BASE_SEED, model="unconstrained")
    add("single agent evacuates", "unconstrained", int(np.isfinite(single).sum()),
        np.isfinite(single).sum() == 1)

    for m in MODELS:
        sub = runs[runs["model"] == m]
        add("evacuated count never exceeds crowd size", m,
            int(sub["n_evacuated"].max()), (sub["n_evacuated"] <= sub["n_agents"]).all())
        add("all evacuation times within horizon", m,
            float(sub["last_evacuation_s"].max()), (sub["last_evacuation_s"].dropna() <= T_MAX).all())

    # Capacity model: the exit cannot pass more people than its flow allows
    cap = runs[runs["model"] == "capacity"].copy()
    cap["capacity_limit"] = EXIT_FLOW_J * cap["exit_width_m"] * cap["last_evacuation_s"] + 1.0
    ok = (cap["n_evacuated"] <= cap["capacity_limit"] + 1e-9).all()
    add("capacity model respects exit flow limit", "capacity", int(ok), ok,
        "holds by construction, because the queue serves at the assumed flow rate")

    # Unconstrained model: informational, shows the physical limit it does not impose
    unc = runs[runs["model"] == "unconstrained"].copy()
    unc["capacity_limit"] = EXIT_FLOW_J * unc["exit_width_m"] * unc["last_evacuation_s"] + 1.0
    viol = (unc["n_evacuated"] > unc["capacity_limit"] + 1e-9)
    add("unconstrained model exceeds the assumed exit flow limit (share of runs)",
        "unconstrained", float(viol.mean()), True,
        "informational: this is the known limitation the capacity model addresses")

    out = pd.DataFrame(checks)
    return out


In [6]:
# ----------------------------------------------------------------------------
# 5. Statistics
# ----------------------------------------------------------------------------
def bootstrap_median_ci(x, n_boot=5000, seed=1, alpha=0.05):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    meds = np.median(x[idx], axis=1)
    return float(np.quantile(meds, alpha / 2)), float(np.quantile(meds, 1 - alpha / 2))


def holm(pvals):
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj


def paired_test(df, factor, low, high, model, label):
    """Paired comparison of high minus low, matched on all other factors and replicate."""
    keys = [f for f in ["n_agents", "exit_width_m", "desired_speed_mps", "obstacle"] if f != factor]
    keys += ["replicate"]
    sub = df[df["model"] == model]
    a = sub[sub[factor] == low].set_index(keys)["t90_s"].rename("low")
    b = sub[sub[factor] == high].set_index(keys)["t90_s"].rename("high")
    joined = pd.concat([a, b], axis=1, join="inner").dropna()
    d = (joined["high"] - joined["low"]).values
    _, p = stats.wilcoxon(d, alternative="two-sided", zero_method="wilcox")
    lo, hi = bootstrap_median_ci(d)
    return {
        "model": model, "hypothesis": label,
        "comparison": f"{factor}: {high} minus {low}", "n_pairs": int(len(d)),
        "median_difference_s": float(np.median(d)),
        "ci_low_s": lo, "ci_high_s": hi,
        "wilcoxon_p": float(p),
    }


def run_tests(runs):
    specs = [
        ("exit_width_m", 0.8, 1.6, "H1: wider exits reduce t90"),
        ("n_agents", 40, 80, "H2: larger crowds increase t90"),
        ("obstacle", 0, 1, "H3: obstacle worsens t90"),
    ]
    rows = [paired_test(runs, f, lo, hi, m, lab) for m in MODELS for f, lo, hi, lab in specs]
    adj = holm([r["wilcoxon_p"] for r in rows])
    for r, p_adj in zip(rows, adj):
        r["holm_p_all_six"] = float(p_adj)
    return pd.DataFrame(rows)


def factorial_regression(df):
    """OLS on coded factors with main effects and two-way interactions for one model."""
    width_c = (df["exit_width_m"] - 1.2) / 0.4
    crowd_c = np.where(df["n_agents"] == 80, 1.0, -1.0)
    speed_c = np.where(df["desired_speed_mps"] == 1.3, 1.0, -1.0)
    obst_c = np.where(df["obstacle"] == 1, 1.0, -1.0)
    names = ["intercept", "width", "crowd", "speed", "obstacle",
             "width:crowd", "width:speed", "width:obstacle",
             "crowd:speed", "crowd:obstacle", "speed:obstacle"]
    X = np.column_stack([
        np.ones(len(df)), width_c, crowd_c, speed_c, obst_c,
        width_c * crowd_c, width_c * speed_c, width_c * obst_c,
        crowd_c * speed_c, crowd_c * obst_c, speed_c * obst_c,
    ])
    y = df["t90_s"].values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    sigma2 = (resid @ resid) / dof
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
    t_vals = beta / se
    p_vals = 2 * (1 - stats.t.cdf(np.abs(t_vals), dof))
    return pd.DataFrame({"term": names, "coefficient_s": beta, "std_error": se,
                         "t": t_vals, "p_value": p_vals})


In [7]:
# ----------------------------------------------------------------------------
# 6. Figures and outputs
# ----------------------------------------------------------------------------
def make_figure(df, out_path):
    fig, axes = plt.subplots(len(MODELS), len(CROWD_SIZES), figsize=(10, 7.5), sharey=True)
    for i, m in enumerate(MODELS):
        for j, crowd in enumerate(CROWD_SIZES):
            ax = axes[i, j]
            sub = df[(df["model"] == m) & (df["n_agents"] == crowd)]
            data = [sub[sub["exit_width_m"] == w]["t90_s"].values for w in EXIT_WIDTHS]
            ax.boxplot(data, tick_labels=[f"{w:.1f} m" for w in EXIT_WIDTHS])
            ax.set_title(f"{m}, {crowd} agents", fontsize=10)
            ax.grid(axis="y", alpha=0.3)
            if j == 0:
                ax.set_ylabel("Time to evacuate 90% (s)")
            if i == len(MODELS) - 1:
                ax.set_xlabel("Exit width")
    fig.suptitle("Simulated time to 90% evacuation (synthetic output, not validated)", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def make_sensitivity_figure(sens, out_path):
    fig, ax = plt.subplots(figsize=(8, 4))
    labels, y = [], []
    for k, (_, row) in enumerate(sens.iterrows()):
        labels.append(f"{row['variant']} ({row['model']})")
        y.append(k)
        ax.errorbar(row["median_width_effect_s"], k,
                    xerr=[[row["median_width_effect_s"] - row["ci_low_s"]],
                          [row["ci_high_s"] - row["median_width_effect_s"]]],
                    fmt="o", color="teal", capsize=3)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Median effect of widening exit 0.8 m to 1.6 m on t90 (s), 95% CI")
    ax.set_title("Sensitivity of the width effect to interaction parameters")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main():
    out = Path("results")
    out.mkdir(exist_ok=True)

    n_conditions = len(MODELS) * len(CROWD_SIZES) * len(EXIT_WIDTHS) * len(DESIRED_SPEEDS) * len(OBSTACLE)
    print(f"Main experiment: {n_conditions} conditions x {REPLICATES} replicates "
          f"= {n_conditions * REPLICATES} simulations")
    runs = run_experiment()
    runs.to_csv(out / "evacuation_simulation_replicates.csv", index=False)

    summary = (runs.groupby(["model", "n_agents", "exit_width_m", "desired_speed_mps", "obstacle"])
               .agg(runs=("t90_s", "size"),
                    t90_mean_s=("t90_s", "mean"),
                    t90_sd_s=("t90_s", "std"),
                    fraction_evacuated_mean=("fraction_evacuated", "mean"),
                    throughput_mean_per_s=("throughput_per_s", "mean"),
                    censored_runs=("t90_censored", "sum"))
               .reset_index())
    summary.to_csv(out / "scenario_summary.csv", index=False)

    print("Verification checks...")
    verification = run_verification(runs)
    verification.to_csv(out / "verification_checks.csv", index=False)

    print("Hypothesis tests...")
    tests = run_tests(runs)
    tests.to_csv(out / "hypothesis_tests.csv", index=False)

    print("Factorial regressions...")
    regs = []
    for m in MODELS:
        r = factorial_regression(runs[runs["model"] == m])
        r.insert(0, "model", m)
        regs.append(r)
    pd.concat(regs).to_csv(out / "factorial_regression.csv", index=False)

    print("Sensitivity analysis...")
    sens = run_sensitivity()
    sens.to_csv(out / "sensitivity_width_effect.csv", index=False)

    make_figure(runs, out / "t90_by_condition.png")
    make_sensitivity_figure(sens, out / "sensitivity_width_effect.png")

    meta = {
        "description": "Synthetic agent-based pedestrian evacuation simulation, version 2",
        "validated_against_real_data": False,
        "models": MODELS, "exit_flow_J_persons_per_m_per_s": EXIT_FLOW_J,
        "room_m": ROOM, "exit_centre_m": [EXIT_X, ROOM],
        "crowd_sizes": CROWD_SIZES, "exit_widths_m": EXIT_WIDTHS,
        "desired_speeds_mps": DESIRED_SPEEDS, "obstacle_conditions": OBSTACLE,
        "replicates": REPLICATES, "base_seed": BASE_SEED,
        "repulsion_mps": REPULSION, "sep_radius_m": SEP_RADIUS,
        "dt_s": DT, "t_max_s": T_MAX,
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    (out / "run_metadata.json").write_text(json.dumps(meta, indent=2))

    print("Done. Outputs in ./results")
    print(verification.to_string(index=False))
    print(tests.round(4).to_string(index=False))
    print(sens.round(3).to_string(index=False))


## 2. Run the experiment, verification, statistics and sensitivity analysis

In [8]:
main()


Main experiment: 48 conditions x 20 replicates = 960 simulations


Verification checks...
Hypothesis tests...
Factorial regressions...


Sensitivity analysis...


Done. Outputs in ./results
                                                                  check         model     value  passed                                                                     note
                               reproducibility (same seed, same output)      capacity      True    True                                                                         
                                                 single agent evacuates unconstrained         1    True                                                                         
                               evacuated count never exceeds crowd size unconstrained        80    True                                                                         
                                    all evacuation times within horizon unconstrained      20.0    True                                                                         
                               evacuated count never exceeds crowd size      capacity   